# Phase 3 — Machine Learning Modeling
## Healthcare AI System

**Input:**  `outputs/model_table.csv` — 25,000 rows  
**Output:** 
- `models/risk_model.joblib`  
- `models/claim_model.joblib`  
- `outputs/feature_schema.json`

**Two Models:**
- **Model A** — Visit Risk Classification (Low / Medium / High)
- **Model B** — Claim Outcome Prediction (Paid / Pending / Rejected)

In [1]:
# Imports
import pandas as pd
import numpy as np
import json
import joblib
import warnings
warnings.filterwarnings("ignore")

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (classification_report,
                             confusion_matrix,
                             accuracy_score,
                             f1_score)
from sklearn.model_selection import RandomizedSearchCV
from xgboost import XGBClassifier

print("Imports done ✓")

Imports done ✓


In [2]:
# Load the data
df = pd.read_csv("../outputs/model_table.csv", parse_dates=["registration_date", "visit_date", "billing_date"])
print(df.shape)
df.head()

(25000, 30)


,patient_id,age,gender,city,insurance_provider,chronic_flag,registration_date,visit_id,visit_date,department,...,risk_score_encoded,claim_status_encoded,is_rejected,days_since_registration,visit_frequency,avg_los_per_patient,provider_rejection_rate,visit_month,visit_dayofweek,high_cost_visit_flag
0,2,15,F,Mumbai,CareOne,0,2025-12-27,8,2026-01-01,General,...,0,2,1,5,4,21.120000,0.256876,1,3,0
1,12,3,M,Bangalore,CareOne,0,2025-08-13,65,2026-01-01,ICU,...,2,2,1,141,8,23.750000,0.256876,1,3,1
2,129,44,M,Pune,MediCareX,1,2025-07-20,651,2026-01-01,ICU,...,2,1,0,165,3,32.460000,0.242556,1,3,1
3,133,47,F,Delhi,CareOne,1,2025-11-02,670,2026-01-01,General,...,1,0,0,60,3,30.056667,0.256876,1,3,0
4,139,14,F,Chennai,SecureLife,1,2025-02-05,706,2026-01-01,Cardiology,...,1,0,0,330,9,29.030000,0.157496,1,3,1


## Model A — Visit Risk Classification

**Business Purpose:**  
Predict whether a hospital visit represents Low, Medium,
or High operational and clinical risk.

**Target:** `risk_score` — Low / Medium / High  
**Split:**  Time-based — earliest 80% train, latest 20% test  
**Why time-based?** Prevents data leakage from future visits
influencing predictions on past visits.

In [3]:
# Risk features
risk_target = "risk_score"

risk_features = [
    "age",
    "gender",
    "city",
    "insurance_provider",
    "chronic_flag",
    "department",
    "visit_type",
    "doctor_id",
    "length_of_stay_hours",
    "days_since_registration",
    "visit_frequency",
    "avg_los_per_patient",
    "visit_month",
    "visit_dayofweek"
]

# I didn't choose billing features because they are not available at the time of visit. I want to predict risk at the time of visit, not after billing.

risk_df = df[risk_features + [risk_target, "visit_date"]].copy()
risk_df = risk_df.dropna(subset=[risk_target, "visit_date"])

# I dropped rows with missing risk_score and visit_date because they are essential for the model. 
# Missing risk_score means we don't have a target variable, and missing visit_date means we can't determine the time of the visit.
# So we can't split the data into train and test sets based on visit_date if we don't have it.

print("Risk dataset shape: ", risk_df.shape)
risk_df.head()

Risk dataset shape:  (25000, 16)


,age,gender,city,insurance_provider,chronic_flag,department,visit_type,doctor_id,length_of_stay_hours,days_since_registration,visit_frequency,avg_los_per_patient,visit_month,visit_dayofweek,risk_score,visit_date
0,15,F,Mumbai,CareOne,0,General,OPD,105,9.63,5,4,21.120000,1,3,Low,2026-01-01
1,3,M,Bangalore,CareOne,0,ICU,ICU,112,59.60,141,8,23.750000,1,3,High,2026-01-01
2,44,M,Pune,MediCareX,1,ICU,ER,150,59.28,165,3,32.460000,1,3,High,2026-01-01
3,47,F,Delhi,CareOne,1,General,OPD,145,25.15,60,3,30.056667,1,3,Medium,2026-01-01
4,14,F,Chennai,SecureLife,1,Cardiology,ER,148,42.88,330,9,29.030000,1,3,Medium,2026-01-01


In [4]:
# Time-based split
risk_df = risk_df.sort_values("visit_date").reset_index(drop=True)
split_idx = int(len(risk_df) * 0.8)
risk_train = risk_df.iloc[:split_idx].copy()
risk_test = risk_df.iloc[split_idx:].copy()

X_train_risk = risk_train[risk_features]
y_train_risk = risk_train[risk_target]
X_test_risk = risk_test[risk_features]
y_test_risk = risk_test[risk_target]

print("Training set shape: ", X_train_risk.shape)
print("Test set shape: ", X_test_risk.shape)
print(f"Train Period: {risk_train['visit_date'].min().date()} to {risk_train['visit_date'].max().date()}")
print(f"Test Period: {risk_test['visit_date'].min().date()} to {risk_test['visit_date'].max().date()}")

Training set shape:  (20000, 14)
Test set shape:  (5000, 14)
Train Period: 2025-01-21 to 2026-01-02
Test Period: 2026-01-02 to 2026-01-20


### Preprocessing Pipeline

Two transformers:
- **Numeric** → SimpleImputer (median strategy)
- **Categorical** → SimpleImputer + OneHotEncoder

Combined using ColumnTransformer.
This entire pipeline gets saved with the model — 
no separate preprocessing step at inference time.

In [5]:
# Preprocessing Pipeline
risk_numeric_features = [
    "age", "chronic_flag", "length_of_stay_hours",
    "days_since_registration", "visit_frequency",
    "doctor_id", "avg_los_per_patient",
    "visit_month", "visit_dayofweek"
]

risk_categorical_features = [
    "gender", "city", "insurance_provider",
    "department", "visit_type"
]

risk_numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

risk_categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

risk_preprocessor = ColumnTransformer(transformers=[
    ("num", risk_numeric_transformer,    risk_numeric_features),
    ("cat", risk_categorical_transformer, risk_categorical_features)
])

print("Preprocessing pipeline ready ✓")

Preprocessing pipeline ready ✓


### Baseline Model: Logistic Regression

We always start with the simplest model.
Logistic Regression gives us a baseline to beat.
`class_weight='balanced'` — handles class imbalance
as proven in Phase 2 EDA.

In [6]:
# Logistic Regression Model
risk_baseline_model = Pipeline(steps=[
    ("preprocessor", risk_preprocessor),
    ("classifier", LogisticRegression(max_iter=1000, random_state=42, class_weight="balanced"))
])

# Train the model
risk_baseline_model.fit(X_train_risk, y_train_risk)

# Predict on the test set
risk_baseline_predictions_train = risk_baseline_model.predict(X_train_risk)
risk_baseline_predictions_test = risk_baseline_model.predict(X_test_risk)

baseline_acc = accuracy_score(y_test_risk, risk_baseline_predictions_test)
print("Risk Baseline Train Accuracy :", round(accuracy_score(y_train_risk, risk_baseline_predictions_train), 4))
print("Risk Baseline Test Accuracy  :", round(accuracy_score(y_test_risk,  risk_baseline_predictions_test), 4))
print("Risk Baseline Test Weighted F1:", round(f1_score(y_test_risk, risk_baseline_predictions_test, average="weighted"), 4))
print("\nClassification Report:\n")
print(classification_report(y_test_risk, risk_baseline_predictions_test))
print("Confusion Matrix:\n", confusion_matrix(y_test_risk, risk_baseline_predictions_test))

Risk Baseline Train Accuracy : 0.9195
Risk Baseline Test Accuracy  : 0.9076
Risk Baseline Test Weighted F1: 0.9082

Classification Report:

              precision    recall  f1-score   support

        High       0.89      0.96      0.92       838
         Low       0.96      0.90      0.93      2360
      Medium       0.85      0.90      0.88      1802

    accuracy                           0.91      5000
   macro avg       0.90      0.92      0.91      5000
weighted avg       0.91      0.91      0.91      5000

Confusion Matrix:
 [[ 801    0   37]
 [   0 2118  242]
 [ 104   79 1619]]


### Advanced Model: Random Forest

Random Forest captures **non-linear interactions** across
patient, department, and visit characteristics.

Logistic Regression assumes linear relationships.
Hospital risk is rarely linear —
a 70-year-old chronic ICU patient is not just
the sum of those individual risk factors.

Key parameters:
- `class_weight='balanced_subsample'` — balances per tree
- `max_depth=8` — prevents overfitting
- `min_samples_leaf=10` — minimum samples at leaf node

Example:
- Doctor 1  →  High
- Doctor 2  →  High
- Doctor 3  →  Medium
- Doctor 4  →  High
- ...
Final vote  →  High  ✅

In [7]:
# Random Forest Model
risk_rf_model = Pipeline(steps=[
    ("preprocessor", risk_preprocessor),
    ("classifier", RandomForestClassifier(
        n_estimators=200,
        max_depth=8,
        min_samples_split=20,
        min_samples_leaf=10,
        class_weight="balanced_subsample",
        random_state=42
    ))
])

risk_rf_model.fit(X_train_risk, y_train_risk)

risk_rf_pred_train = risk_rf_model.predict(X_train_risk)
risk_rf_pred_test  = risk_rf_model.predict(X_test_risk)

rf_acc = accuracy_score(y_test_risk, risk_rf_pred_test)

print("Risk RF Train Accuracy :", round(accuracy_score(y_train_risk, risk_rf_pred_train), 4))
print("Risk RF Test Accuracy  :", round(accuracy_score(y_test_risk,  risk_rf_pred_test),  4))
print("Risk RF Test Weighted F1:", round(f1_score(y_test_risk, risk_rf_pred_test, average="weighted"), 4))
print("\nClassification Report:\n")
print(classification_report(y_test_risk, risk_rf_pred_test))
print("Confusion Matrix:\n", confusion_matrix(y_test_risk, risk_rf_pred_test))

Risk RF Train Accuracy : 0.9414
Risk RF Test Accuracy  : 0.935
Risk RF Test Weighted F1: 0.935

Classification Report:

              precision    recall  f1-score   support

        High       0.95      0.93      0.94       838
         Low       0.95      0.95      0.95      2360
      Medium       0.91      0.91      0.91      1802

    accuracy                           0.94      5000
   macro avg       0.94      0.93      0.93      5000
weighted avg       0.94      0.94      0.94      5000

Confusion Matrix:
 [[ 779    0   59]
 [   0 2253  107]
 [  44  115 1643]]


### XGBoost Model

XGBoost uses **gradient boosting** — builds trees sequentially,
each correcting the errors of the previous one.

Why XGBoost here:
- Handles mixed numeric + categorical features well
- Built-in regularisation prevents overfitting
- Typically outperforms Random Forest on structured tabular data

**Note:** XGBoost requires numeric labels.
We use LabelEncoder to convert Low/Medium/High → 0/1/2.
We keep separate variables (y_train_risk_xgb / y_test_risk_xgb)
so LR and RF evaluations above are not affected.

In [8]:
# Encode target for XGBoost only
le = LabelEncoder()
risk_df["risk_score_label_encoded"] = le.fit_transform(risk_df["risk_score"])
print("Label Mapping: ", dict(zip(le.classes_, le.transform(le.classes_))))

Label Mapping:  {'High': np.int64(0), 'Low': np.int64(1), 'Medium': np.int64(2)}


In [9]:
# Time-based split for XGBoost
# Using same split_idx — consistent with LR and RF
risk_df = risk_df.sort_values("visit_date").reset_index(drop=True)
split_idx = int(len(risk_df) * 0.8)

risk_train_xgb = risk_df.iloc[:split_idx].copy()
risk_test_xgb  = risk_df.iloc[split_idx:].copy()

X_train_risk = risk_train_xgb[risk_features]
y_train_risk_xgb = risk_train_xgb["risk_score_label_encoded"]
X_test_risk  = risk_test_xgb[risk_features]
y_test_risk_xgb  = risk_test_xgb["risk_score_label_encoded"]

print("XGBoost train shape:", X_train_risk.shape)
print("XGBoost test shape :", X_test_risk.shape)

XGBoost train shape: (20000, 14)
XGBoost test shape : (5000, 14)


In [10]:
# XGBoost Model
risk_xgb_model = Pipeline(steps=[
    ("preprocessor", risk_preprocessor),
    ("classifier", XGBClassifier(
        objective="multi:softmax",
        num_class=3,
        n_estimators=500,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        verbosity=0
    ))
])

# Train → Predict → Evaluate
# Train
risk_xgb_model.fit(X_train_risk, y_train_risk_xgb)

# Predict Train
risk_xgb_pred_train = risk_xgb_model.predict(X_train_risk)
# Predict Test
risk_xgb_pred_test  = risk_xgb_model.predict(X_test_risk)

# Save for comparison
xgb_acc = accuracy_score(y_test_risk_xgb, risk_xgb_pred_test)

print("Risk XGB Train Accuracy :", round(accuracy_score(y_train_risk_xgb, risk_xgb_pred_train), 4))
print("Risk XGB Test Accuracy  :", round(accuracy_score(y_test_risk_xgb,  risk_xgb_pred_test),  4))
print("Risk XGB Test Weighted F1:", round(f1_score(y_test_risk_xgb, risk_xgb_pred_test, average="weighted"), 4))
print("\nClassification Report:\n")
print(classification_report(y_test_risk_xgb, risk_xgb_pred_test,
                             target_names=le.classes_))
print("Confusion Matrix:\n", confusion_matrix(y_test_risk_xgb, risk_xgb_pred_test))

Risk XGB Train Accuracy : 0.9941
Risk XGB Test Accuracy  : 0.955
Risk XGB Test Weighted F1: 0.9551

Classification Report:

              precision    recall  f1-score   support

        High       0.98      0.94      0.96       840
         Low       0.97      0.97      0.97      2360
      Medium       0.93      0.95      0.94      1800

    accuracy                           0.95      5000
   macro avg       0.96      0.95      0.95      5000
weighted avg       0.96      0.95      0.96      5000

Confusion Matrix:
 [[ 790    0   50]
 [   0 2278   82]
 [  16   77 1707]]


In [11]:
# Model Comparison
print("=" * 45)
print("MODEL A — RISK CLASSIFICATION COMPARISON")
print("=" * 45)
print(f"Logistic Regression : {baseline_acc:.4f}")
print(f"Random Forest       : {rf_acc:.4f}")
print(f"XGBoost             : {xgb_acc:.4f}")
print()
print(f"Best model: XGBoost → {max(baseline_acc, rf_acc, xgb_acc):.4f}")

MODEL A — RISK CLASSIFICATION COMPARISON
Logistic Regression : 0.9076
Random Forest       : 0.9350
XGBoost             : 0.9550

Best model: XGBoost → 0.9550


### Optional - Hyperparameter Tuning: Random Forest - Not needed at the moment

We use RandomizedSearchCV to find the best RF parameters.
`f1_weighted` is our scoring metric — not accuracy.
This ensures minority classes (High risk) are weighted properly.

In [12]:
rf_param_grid = {
    "classifier__n_estimators":   [200, 300, 400],
    "classifier__max_depth":      [8, 12, 16, None],
    "classifier__min_samples_split": [5, 10, 20],
    "classifier__min_samples_leaf":  [1, 2, 4]
}

rf_random_search = RandomizedSearchCV(
    risk_rf_model,           # the pipeline to tune
    param_distributions=rf_param_grid,  # values to try
    n_iter=10,               # try only 10 random combinations
    cv=3,                    # 3-fold cross validation
    scoring="f1_weighted",   # optimise for weighted F1
    n_jobs=-1,               # use all CPU cores
    random_state=42          # reproducible random selection
)

rf_random_search.fit(X_train_risk, y_train_risk)

print("Best Parameters:", rf_random_search.best_params_)

Best Parameters: {'classifier__n_estimators': 200, 'classifier__min_samples_split': 5, 'classifier__min_samples_leaf': 4, 'classifier__max_depth': 12}


In [13]:
best_rf_model  = rf_random_search.best_estimator_
rf_tuned_pred  = best_rf_model.predict(X_test_risk)

print("Tuned RF Accuracy   :", round(accuracy_score(y_test_risk, rf_tuned_pred), 4))
print("Tuned RF Weighted F1:", round(f1_score(y_test_risk, rf_tuned_pred, average="weighted"), 4))
print("\nClassification Report:\n")
print(classification_report(y_test_risk, rf_tuned_pred))

Tuned RF Accuracy   : 0.435
Tuned RF Weighted F1: 0.3815

Classification Report:

              precision    recall  f1-score   support

        High       0.23      0.05      0.08       838
         Low       0.47      0.74      0.57      2360
      Medium       0.36      0.22      0.27      1802

    accuracy                           0.43      5000
   macro avg       0.35      0.33      0.31      5000
weighted avg       0.39      0.43      0.38      5000



In [ ]:
#Takes 20,000 training rows
#Randomly shuffles them
#Splits into 3 folds

#Fold 1: rows 3, 7, 12, 19, 25...   ← random mix of all dates
#Fold 2: rows 1, 4, 9, 14, 22...    ← random mix of all dates
#Fold 3: rows 2, 5, 8, 11, 17...    ← random mix of all dates

In [ ]:
from sklearn.model_selection import TimeSeriesSplit
tscv = TimeSeriesSplit(n_splits=3)
# Train on past → validate on future — every fold
# Fold 1: train Jan-Apr,  validate May
# Fold 2: train Jan-Jul,  validate Aug
# Fold 3: train Jan-Oct,  validate Nov

---
## Model B — Claim Outcome Prediction

**Business Purpose:**  
Predict whether an insurance claim will be Paid, Pending,
or Rejected — before submission.

**Target:** `claim_status` — Paid / Pending / Rejected  
**Split:**  Time-based on `billing_date`

**Leakage prevention:**  
`approved_amount` and `payment_days` excluded —
both are post-outcome variables that are only known
AFTER the claim is processed.